# Bitácora de cambios

## Actividades realizadas

- Crea módulo de Alice en PyTorch en `src/alice.py`.
- Ajustada la arquitectura para seguir la versión original de Abadi & Andersen (2016):
  - Entrada: concatenación mensaje + clave → vector `2N`.
  - Capa fully connected de tamaño `2N × 2N`.
  - Cuatro convoluciones 1D con ventanas y strides correspondientes a `[4,1,2]`, `[2,2,4]`, `[1,4,4]`, `[1,4,1]`.
  - Salida final de tamaño `N` con activación `tanh`.
- Corregido el `padding` en PyTorch para mantener la reducción de longitud intencional y evitar errores en capas con stride.
- Verificado que el script `src/alice.py` se ejecuta correctamente y produce salida de tamaño `[batch_size, N]`.

## Notas

-Cometí un error al crear el workspace, me trajo anc y cryptography, pero esas son otras carpetas de codigo que no deben estar aquí. Las sacaré luego. Necesito documentarme para no romper nada d enuevo.
- El repositorio Git detecta el directorio `AdversarialTracrProject` como un nuevo conjunto de archivos no rastreados.
- Este commit incluye tanto el código actualizado como el archivo de bitácora.

27-06-26 he recreado lo más fiel posible la arquitectura de Abadi & Andersen (2016) en PyTorch, ajustando las capas convolucionales y el padding para mantener la reducción de tamaño intencional. El script `src/alice.py` ahora se ejecuta correctamente y produce la salida esperada. Para XAI necesito avanzar en la comprensión de los xor para luego pasar a programar el resultado de eve en tracr, pero aún no he llegado a esa etapa.
Debo terminar de leer "think like a transformers".
! idea, buscaré alguna implementación RASP ya hecha en ggithub respecto de los xor, para ver si puedo usarla como referencia para mi implementación de Eve en Tracr.

### 05/07/26 
Alice es la red "emisora" en este esquema de criptografía neuronal adversarial (estilo Abadi & Andersen). Dado un mensaje y una clave compartida (cada uno de msg_length bits), aprende a producir un texto cifrado que:

Bob pueda descifrar — Bob recibe el cifrado + la clave y debe reconstruir el mensaje original (se penaliza bob_loss si falla).
Eve NO pueda descifrar — Eve solo ve el cifrado (sin la clave) y trata de adivinar el mensaje; Alice es recompensada cuando Eve falla (se resta alpha * eve_loss en su función de pérdida, alice.py:113).
Se parezca a un XOR explícito — hay un término extra beta * xor_loss (alice.py:110) que empuja a Alice a converger hacia el cifrado XOR bit a bit de msg y key, es decir, hacia un "one-time pad" aprendido, en vez de solo cualquier función arbitraria que engañe a Eve.
Arquitectura: concatena msg y key (2×N bits) → capa oculta ReLU → capa de salida sigmoide de N bits (alice.py:38-41), así que el cifrado es continuo en [0,1] (se binariza con umbral 0.5 al evaluar).

En resumen: Alice aprende una función de cifrado condicionada por la clave que minimiza el error de Bob, maximiza el error de Eve, y se acerca al comportamiento de un XOR ideal.
# Bitácora de cambios

## Actividades realizadas

- Crea módulo de Alice en PyTorch en `src/alice.py`.
- Ajustada la arquitectura para seguir la versión original de Abadi & Andersen (2016):
  - Entrada: concatenación mensaje + clave → vector `2N`.
  - Capa fully connected de tamaño `2N × 2N`.
  - Cuatro convoluciones 1D con ventanas y strides correspondientes a `[4,1,2]`, `[2,2,4]`, `[1,4,4]`, `[1,4,1]`.
  - Salida final de tamaño `N` con activación `tanh`.
- Corregido el `padding` en PyTorch para mantener la reducción de longitud intencional y evitar errores en capas con stride.
- Verificado que el script `src/alice.py` se ejecuta correctamente y produce salida de tamaño `[batch_size, N]`.

## Notas

-Cometí un error al crear el workspace, me trajo anc y cryptography, pero esas son otras carpetas de codigo que no deben estar aquí. Las sacaré luego. Necesito documentarme para no romper nada d enuevo.
- El repositorio Git detecta el directorio `AdversarialTracrProject` como un nuevo conjunto de archivos no rastreados.
- Este commit incluye tanto el código actualizado como el archivo de bitácora.

27-06-26 he recreado lo más fiel posible la arquitectura de Abadi & Andersen (2016) en PyTorch, ajustando las capas convolucionales y el padding para mantener la reducción de tamaño intencional. El script `src/alice.py` ahora se ejecuta correctamente y produce la salida esperada. Para XAI necesito avanzar en la comprensión de los xor para luego pasar a programar el resultado de eve en tracr, pero aún no he llegado a esa etapa.
Debo terminar de leer "think like a transformers".
! idea, buscaré alguna implementación RASP ya hecha en ggithub respecto de los xor, para ver si puedo usarla como referencia para mi implementación de Eve en Tracr.

### 05/07/26 
Alice es la red "emisora" en este esquema de criptografía neuronal adversarial (estilo Abadi & Andersen). Dado un mensaje y una clave compartida (cada uno de msg_length bits), aprende a producir un texto cifrado que:

Bob pueda descifrar — Bob recibe el cifrado + la clave y debe reconstruir el mensaje original (se penaliza bob_loss si falla).
Eve NO pueda descifrar — Eve solo ve el cifrado (sin la clave) y trata de adivinar el mensaje; Alice es recompensada cuando Eve falla (se resta alpha * eve_loss en su función de pérdida, alice.py:113).
Se parezca a un XOR explícito — hay un término extra beta * xor_loss (alice.py:110) que empuja a Alice a converger hacia el cifrado XOR bit a bit de msg y key, es decir, hacia un "one-time pad" aprendido, en vez de solo cualquier función arbitraria que engañe a Eve.
Arquitectura: concatena msg y key (2×N bits) → capa oculta ReLU → capa de salida sigmoide de N bits (alice.py:38-41), así que el cifrado es continuo en [0,1] (se binariza con umbral 0.5 al evaluar).

En resumen: Alice aprende una función de cifrado condicionada por la clave que minimiza el error de Bob, maximiza el error de Eve, y se acerca al comportamiento de un XOR ideal.

Nota al margen: encontré un bug en la firma de train_anc — alice.py:79 tiene ba#tch_size (el # convierte el resto de la línea en comentario, dejando un parámetro ba colgado). Puede que quisieras batch_size: int | None = None,. ¿Quieres que lo corrija?
Nota al margen: encontré un bug en la firma de train_anc — alice.py:79 tiene ba#tch_size (el # convierte el resto de la línea en comentario, dejando un parámetro ba colgado). Puede que quisieras batch_size: int | None = None,. ¿Quieres que lo corrija?